In [1]:
import os
import glob
import json
import numpy as np
import pandas as pd
from collections import Counter

# ------------------------------------------------------------
# Partie 1 : Extraction du vocabulaire des personnages
# ------------------------------------------------------------

In [2]:
def read_booknlp(path_book):
    with open(path_book, "r", encoding="utf-8") as file:
        lines = file.readlines()
        # eval() transforme une chaîne de caractères représentant un dictionnaire en objet Python
        dicts = [eval(line.strip()) for line in lines if line.strip()]
    return dicts[:5] # /!\ On ne prends que les 5 premiers personnages pour un roman

In [3]:
def get_characterization(booknlp_data):
    list_tout_mots = []
    for i in range(len(booknlp_data)):
        list_mots_par_perso = []
        list_mots_par_perso.extend([item['w'] for item in booknlp_data[i]['agent']])
        list_mots_par_perso.extend([item['w'] for item in booknlp_data[i]['patient']])
        list_mots_par_perso.extend([item['w'] for item in booknlp_data[i]['poss']])
        list_mots_par_perso.extend([item['w'] for item in booknlp_data[i]['mod']])
        list_tout_mots.append(list_mots_par_perso)
        
    return list_tout_mots

In [6]:
book_files = glob.glob(os.path.join("/Users/deniseatzori/Library/Mobile Documents/com~apple~CloudDocs/ENC-PSL/Memoire/Github_tesi/Corpus_propp", "*.book"))

In [ ]:
# Extraction de tous les mots des personnages
all_characters_words = []
for filepath in book_files:
    print(filepath)
    book_data = read_booknlp(filepath)
    characters_words = get_characterization(book_data)
    # Pour chaque personnage, on ajoute ses mots dans la liste globale
    for list_word in characters_words:
        for word in list_word:
            all_characters_words.append(word)

In [8]:
# Calcul des 1000 mots les plus fréquents parmi les mots extraits des personnages
counter_characters = Counter(all_characters_words)
most_common_items = counter_characters.most_common(1000)

top_1000_words = []
for item in most_common_items:
    top_1000_words.append(item[0])

print("Nombre total de mots extraits des personnages :", len(all_characters_words))
print("Exemple des 10 premiers MFW :", top_1000_words[:10])

Nombre total de mots extraits des personnages : 489638
Exemple des 10 premiers MFW : ['avoir', 'dire', 'voir', 'faire', 'vouloir', 'pouvoir', 'aller', 'petit', 'savoir', 'aimer']


# ------------------------------------------------------------
# Partie 2 : Construction du BoW pour les personnages
# ------------------------------------------------------------

In [9]:
bow_characters = {}  # Dictionary to store the BoW for each character

for filepath in book_files:
    filename = os.path.basename(filepath)
    book_data = read_booknlp(filepath)  # Read the 5 first characters
    # Use the updated get_characterization to obtain a list of tokens per character
    characters_tokens = get_characterization(book_data)
    
    for idx, tokens in enumerate(characters_tokens):
        total = len(tokens)
        counter_tokens = Counter(tokens)
        bow = {}

        # add the gender prediction to the bow dictionary
        gender = "Gender"
        if book_data[idx]['gender'].get("argmax") == "Female":
            bow[gender] = 1
        else:
            bow[gender] = 0

        for word in top_1000_words:
            bow[word] = counter_tokens.get(word, 0) / total if total > 0 else 0

        # Try to get the character's name from the 'mentions' field; otherwise, use a default name.
        if "mentions" in book_data[idx] and book_data[idx]["mentions"].get("proper") and len(book_data[idx]["mentions"]["proper"]) > 0:
            char_name = book_data[idx]["mentions"]["proper"][0]["n"].lower()
        else:
            char_name = f"char{idx+1}"

        key = f"{filename} --- {char_name}"
        bow_characters[key] = bow

In [10]:
Gender = ["Gender"]
# Example: Converting to DataFrame and saving as CSV (if desired)
df_bow_characters = pd.DataFrame.from_dict(bow_characters, orient="index", columns= Gender + top_1000_words)

# preparo le colonne per i metadati
df_bow_characters.insert(1, "author_gender", "")
df_bow_characters.insert(2, "type", "")


In [11]:
df_bow_characters.head(5)

,Gender,author_gender,type,avoir,dire,voir,faire,vouloir,pouvoir,aller,...,lunette,gens,façon,propre,ménage,manteau,troupe,futur,daigner,geste
1883_Halt-Marie-Robert_Histoire-d-un-petit-homme.book --- étienne,0,,,0.029769,0.031896,0.017922,0.017618,0.012454,0.007898,0.020049,...,0.000304,0.000304,0.000304,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0
1883_Halt-Marie-Robert_Histoire-d-un-petit-homme.book --- m. duclair,0,,,0.022989,0.054598,0.002874,0.008621,0.008621,0.002874,0.008621,...,0.000000,0.000000,0.000000,0.002874,0.000000,0.0,0.0,0.0,0.0,0.0
1883_Halt-Marie-Robert_Histoire-d-un-petit-homme.book --- thérèse,1,,,0.025316,0.042194,0.008439,0.016878,0.012658,0.012658,0.008439,...,0.008439,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0
1883_Halt-Marie-Robert_Histoire-d-un-petit-homme.book --- cabarratte,0,,,0.040230,0.028736,0.017241,0.005747,0.000000,0.000000,0.005747,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0
1883_Halt-Marie-Robert_Histoire-d-un-petit-homme.book --- florence,1,,,0.017544,0.046784,0.005848,0.000000,0.000000,0.000000,0.011696,...,0.000000,0.000000,0.000000,0.000000,0.005848,0.0,0.0,0.0,0.0,0.0


In [ ]:
df_bow_characters.to_csv("BoW_1000_personnages.csv", index=True)
print("Le fichier 'BoW_1000_personnages.csv' a été sauvegardé.")

Le fichier 'BoW_1000_personnages.csv' a été sauvegardé.


In [32]:
len(df_bow_characters)

614